# Huấn Luyện Matcha-TTS Tiếng Việt Trên Google Colab (VIVOS)

**Đề tài:** Nghiên cứu kiến trúc Optimal-Transport Conditional Flow Matching (Matcha-TTS, ICASSP 2024) và xây dựng mô hình TTS tiếng Việt đa người nói trên bộ dữ liệu VIVOS.

**Cách dùng**
1. `Runtime → Change runtime type → T4 GPU` (hoặc L4 / A100).
2. Chạy lần lượt từ trên xuống (`Runtime → Run all`).
3. Mọi thứ cần giữ lâu dài được lưu trong `MyDrive/Matcha_TTS_Vietnamese/`:
   - `vivos_mels_22k.pt`, `vivos_meta.json`: dữ liệu đã tiền xử lý (tạo **một lần**, các phiên sau không cần tải lại VIVOS).
   - `matcha_vi_resume.pt`: trạng thái huấn luyện mới nhất (model + optimizer). Khi Colab ngắt kết nối, chỉ cần **Run all** lại, quá trình huấn luyện tự tiếp tục.
   - `matcha_vi.pt`: mô hình tốt nhất (validation loss thấp nhất), dùng để sinh giọng ở mục 6.
   - `samples/`: câu nói mẫu sinh ra sau mỗi vài epoch để nghe tiến độ.

## 1. Kiểm Tra GPU & Kết Nối Google Drive

In [ ]:
# 1.1 Kiểm tra GPU: dừng ngay nếu runtime chưa bật GPU
import torch

assert torch.cuda.is_available(), 'Chua bat GPU! Vao Runtime -> Change runtime type -> chon T4 GPU, roi chay lai.'
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv,noheader

gpu = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name} | VRAM {GPU_VRAM_GB:.1f} GB | compute capability {gpu.major}.{gpu.minor}')
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

In [ ]:
# 1.2 Kết nối Google Drive và khai báo đường dẫn lưu trữ
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Matcha_TTS_Vietnamese'
DATA_PACK = f'{DRIVE_DIR}/vivos_mels_22k.pt'    # mel-spectrogram đã tiền xử lý (~400 MB)
META_PATH = f'{DRIVE_DIR}/vivos_meta.json'      # vocab, danh sách người nói, thống kê mel
RESUME_CKPT = f'{DRIVE_DIR}/matcha_vi_resume.pt'  # trạng thái huấn luyện mới nhất (để resume)
MODEL_CKPT = f'{DRIVE_DIR}/matcha_vi.pt'          # mô hình tốt nhất theo validation loss
SAMPLES_DIR = f'{DRIVE_DIR}/samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

# Đổi tên checkpoint của bản notebook trước (nếu có) để vẫn resume được
for old_name, new_path in (('vivos_last.pt', RESUME_CKPT), ('vivos_best.pt', MODEL_CKPT)):
    if os.path.exists(f'{DRIVE_DIR}/{old_name}') and not os.path.exists(new_path):
        os.rename(f'{DRIVE_DIR}/{old_name}', new_path)
        print(f'[RENAME] {old_name} -> {os.path.basename(new_path)}')
print(f'[OK] Thu muc luu tru tren Drive: {DRIVE_DIR}')

## 2. Cài Đặt Matcha-TTS

Không `pip install` Matcha-TTS (bản gốc cần biên dịch Cython, lỗi trên Python 3.13). Thay vào đó: clone mã nguồn, thêm vào `sys.path`, chỉ cài các thư viện mà phần mô hình cần, và thay Monotonic Alignment Search (MAS) bằng bản biên dịch **numba**.

In [ ]:
import os
import sys

MATCHA_DIR = '/content/Matcha-TTS'
if not os.path.exists(MATCHA_DIR):
    !git clone -q --depth 1 https://github.com/shivammehta25/Matcha-TTS.git {MATCHA_DIR}
!pip install -q lightning hydra-core conformer==0.3.2 diffusers einops wget gdown numba

# Monotonic Alignment Search viết bằng numba, thay cho C-extension Cython của bản gốc
MAS_CODE = """import numba
import numpy as np
import torch


@numba.njit(parallel=True, cache=True)
def _maximum_path_batch(paths, values, t_xs, t_ys):
    max_neg_val = -1e9
    for b in numba.prange(paths.shape[0]):
        path, value, t_x, t_y = paths[b], values[b], t_xs[b], t_ys[b]
        for y in range(t_y):
            for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
                v_cur = max_neg_val if x == y else value[x, y - 1]
                if x == 0:
                    v_prev = 0.0 if y == 0 else max_neg_val
                else:
                    v_prev = value[x - 1, y - 1]
                value[x, y] += max(v_prev, v_cur)
        index = t_x - 1
        for y in range(t_y - 1, -1, -1):
            path[index, y] = 1
            if index != 0 and (index == y or value[index, y - 1] < value[index - 1, y - 1]):
                index -= 1


def maximum_path(value, mask):
    value = value * mask
    device, dtype = value.device, value.dtype
    value_np = value.detach().float().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()
    path = np.zeros_like(value_np, dtype=np.int32)
    t_x = mask_np.sum(1)[:, 0].astype(np.int32)
    t_y = mask_np.sum(2)[:, 0].astype(np.int32)
    _maximum_path_batch(path, value_np, t_x, t_y)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
"""
with open(f'{MATCHA_DIR}/matcha/utils/monotonic_align/__init__.py', 'w', encoding='utf-8') as f:
    f.write(MAS_CODE)

if MATCHA_DIR not in sys.path:
    sys.path.insert(0, MATCHA_DIR)

from matcha.models.matcha_tts import MatchaTTS  # kiểm tra import
print('[OK] Matcha-TTS san sang.')

## 3. Text Frontend Tiếng Việt & Hàm Dùng Chung

Matcha gốc dùng espeak `en-us`, nên **không đọc được tiếng Việt**. Ở đây mỗi **ký tự tiếng Việt** (chuẩn hóa NFC, chữ thường, nguyên âm mang dấu thanh là một token riêng) được dùng trực tiếp làm token, vì chính tả tiếng Việt khá sát với cách phát âm.

In [ ]:
import re
import unicodedata

import torch
from omegaconf import OmegaConf
from matcha.hifigan.config import v1 as HIFIGAN_V1
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.env import AttrDict
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.models.matcha_tts import MatchaTTS

DEVICE = 'cuda'
# Ampere trở lên (A100, L4) dùng bf16, T4 dùng fp16 + GradScaler
AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Tham số mel giống hệt checkpoint LJSpeech để tái sử dụng decoder và vocoder HiFi-GAN
SAMPLE_RATE, N_FFT, N_MELS, HOP, WIN, FMIN, FMAX = 22050, 1024, 80, 256, 1024, 0, 8000

PRETRAINED_DIR = '/content/pretrained'
MATCHA_LJSPEECH_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/matcha_ljspeech.ckpt'
HIFIGAN_UNIV_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000'


def download(url, filename):
    path = f'{PRETRAINED_DIR}/{filename}'
    if not os.path.exists(path):
        os.makedirs(PRETRAINED_DIR, exist_ok=True)
        torch.hub.download_url_to_file(url, path)
    return path


def normalize_vi(text):
    """Chuẩn hóa NFC, chữ thường, bỏ dấu câu/chữ số (VIVOS không có dấu câu, chưa hỗ trợ đọc số)."""
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'[\W\d_]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def text_to_ids(text, char2id):
    """Văn bản -> chuỗi id, chèn blank (id 0) giữa các token như Matcha gốc (add_blank=True)."""
    ids = [char2id[c] for c in normalize_vi(text) if c in char2id]
    result = [0] * (2 * len(ids) + 1)
    result[1::2] = ids
    return result


def build_model(hparams):
    return MatchaTTS(**OmegaConf.create(hparams))


def load_vocoder():
    # HiFi-GAN universal: tổng quát cho nhiều giọng, phù hợp hơn bản chỉ học giọng LJSpeech
    path = download(HIFIGAN_UNIV_URL, 'hifigan_univ_v1')
    vocoder = HiFiGAN(AttrDict(HIFIGAN_V1)).to(DEVICE)
    vocoder.load_state_dict(torch.load(path, map_location=DEVICE)['generator'])
    vocoder.eval()
    vocoder.remove_weight_norm()
    return vocoder, Denoiser(vocoder, mode='zeros')


@torch.no_grad()
def synthesize_vi(model, text, spk_id, char2id, vocoder, denoiser,
                  n_timesteps=10, temperature=0.667, length_scale=1.0):
    x = torch.tensor(text_to_ids(text, char2id), dtype=torch.long, device=DEVICE)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)
    spks = torch.tensor([spk_id], dtype=torch.long, device=DEVICE)
    was_training = model.training
    model.eval()
    mel = model.synthesise(x, x_lengths, n_timesteps=n_timesteps, temperature=temperature,
                           spks=spks, length_scale=length_scale)['mel']
    model.train(was_training)
    audio = vocoder(mel).clamp(-1, 1)
    return denoiser(audio.squeeze(), strength=0.00025).cpu().squeeze().numpy()


print(f'[OK] AMP dtype: {AMP_DTYPE}. Vi du chuan hoa: {normalize_vi("Xin chào, Việt Nam 2024!")!r}')

## 4. Chuẩn Bị Dữ Liệu VIVOS

- **Lần đầu** (khoảng 10–15 phút): tải VIVOS (1.47 GB), gộp train + test (65 người nói, khoảng 15 giờ), resample 16 kHz → 22.05 kHz, cắt im lặng, tính mel-spectrogram, rồi lưu gói dữ liệu lên Drive.
- **Các lần sau:** chỉ nạp gói dữ liệu từ Drive (khoảng 1 phút), không tải lại VIVOS.

In [ ]:
import json
import random
from multiprocessing import Pool

import librosa
import numpy as np
from tqdm.auto import tqdm
from matcha.utils.audio import mel_spectrogram

VIVOS_URL = 'https://huggingface.co/datasets/AILAB-VNUHCM/vivos/resolve/main/data/vivos.tar.gz'
VIVOS_DIR = '/content/data/vivos'
MAX_SECONDS = 10.0   # bỏ câu dài hơn để tránh tràn VRAM
N_VAL = 100          # số câu giữ lại làm validation


def load_audio(wav_path):
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    y, _ = librosa.effects.trim(y, top_db=40)
    if len(y) < SAMPLE_RATE // 2 or len(y) > MAX_SECONDS * SAMPLE_RATE:
        return None
    return (0.95 * y / max(1e-6, float(np.abs(y).max()))).astype(np.float32)


def read_vivos():
    records, genders = [], {}
    for split in ('train', 'test'):
        split_dir = f'{VIVOS_DIR}/{split}'
        if os.path.exists(f'{split_dir}/genders.txt'):
            with open(f'{split_dir}/genders.txt', encoding='utf-8') as f:
                genders.update(line.split() for line in f if len(line.split()) == 2)
        with open(f'{split_dir}/prompts.txt', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(' ', 1)
                if len(parts) != 2:
                    continue
                utt_id, text = parts
                spk = utt_id.split('_')[0]
                wav_path = f'{split_dir}/waves/{spk}/{utt_id}.wav'
                text = normalize_vi(text)
                if text and os.path.exists(wav_path):
                    records.append({'utt': utt_id, 'spk': spk, 'text': text, 'wav': wav_path})
    return records, genders


if os.path.exists(DATA_PACK) and os.path.exists(META_PATH):
    print('Nap du lieu da tien xu ly tu Google Drive...')
    data = torch.load(DATA_PACK)
    with open(META_PATH, encoding='utf-8') as f:
        meta = json.load(f)
else:
    if not os.path.exists(VIVOS_DIR):
        !mkdir -p /content/data
        !wget -q --show-progress -c -O /content/data/vivos.tar.gz {VIVOS_URL}
        !tar -xzf /content/data/vivos.tar.gz -C /content/data && rm /content/data/vivos.tar.gz
    records, genders = read_vivos()
    print(f'Doc duoc {len(records)} cau tu VIVOS')

    # Đọc + resample + cắt im lặng song song trên CPU, tính mel trên GPU
    items, mel_sum, mel_sq_sum, n_frames = [], 0.0, 0.0, 0
    with Pool(os.cpu_count()) as pool:
        audios = pool.imap(load_audio, [r['wav'] for r in records], chunksize=16)
        for r, y in tqdm(zip(records, audios), total=len(records), desc='Tien xu ly'):
            if y is None:
                continue
            mel = mel_spectrogram(torch.from_numpy(y).unsqueeze(0).to(DEVICE),
                                  N_FFT, N_MELS, SAMPLE_RATE, HOP, WIN, FMIN, FMAX).squeeze(0)
            mel_sum += mel.sum().item()
            mel_sq_sum += (mel ** 2).sum().item()
            n_frames += mel.shape[-1]
            items.append({'utt': r['utt'], 'spk': r['spk'], 'text': r['text'], 'mel': mel.half().cpu()})

    mel_mean = mel_sum / (n_frames * N_MELS)
    meta = {
        'vocab': ['_'] + sorted(set(''.join(it['text'] for it in items))),   # id 0 = blank/pad
        'speakers': sorted(set(it['spk'] for it in items)),
        'genders': genders,
        'mel_mean': mel_mean,
        'mel_std': float(np.sqrt(mel_sq_sum / (n_frames * N_MELS) - mel_mean ** 2)),
    }
    random.Random(1234).shuffle(items)
    data = {'val': items[:N_VAL], 'train': items[N_VAL:]}

    torch.save(data, DATA_PACK)
    with open(META_PATH, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)
    print(f'[SAVE] Da luu goi du lieu len Drive: {DATA_PACK}')

CHAR2ID = {c: i for i, c in enumerate(meta['vocab'])}
SPK2ID = {s: i for i, s in enumerate(meta['speakers'])}
hours = sum(it['mel'].shape[-1] for it in data['train'] + data['val']) * HOP / SAMPLE_RATE / 3600
print(f"[OK] {len(data['train'])} cau train, {len(data['val'])} cau val ({hours:.1f} gio), "
      f"{len(meta['speakers'])} nguoi noi, vocab {len(meta['vocab'])} ky tu")
print(f"     mel_mean={meta['mel_mean']:.4f}, mel_std={meta['mel_std']:.4f}")
print('     Vocab:', ''.join(meta['vocab'][1:]))

## 5. Huấn Luyện Mô Hình

- Matcha-TTS **đa người nói** (65 giọng VIVOS) với vocab ký tự tiếng Việt. Decoder Flow Matching được khởi tạo từ checkpoint LJSpeech; text encoder học lại từ đầu.
- Loss của Matcha: `duration loss + prior loss + flow-matching loss`, dùng mixed precision.
- Mỗi epoch lưu `matcha_vi_resume.pt` (để resume) và `matcha_vi.pt` (mô hình tốt nhất) lên Drive; cứ `SAMPLE_EVERY` epoch phát thử một câu.

> ⚠️ Cần **nhiều giờ** (hàng chục nghìn step) mới có giọng rõ ràng. Hãy theo dõi các câu mẫu để đánh giá tiến độ. Bị ngắt kết nối thì **Run all** lại, quá trình sẽ tự tiếp tục từ epoch cuối.

In [ ]:
import math
import random
import time

import numpy as np
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import Audio, display
from matcha.utils.model import fix_len_compatibility

# ---------------------------- Cấu hình ----------------------------
EPOCHS = 200         # ~385 step/epoch với batch 32 (T4) -> ~77k step
PATIENCE = 15        # dừng sớm nếu validation loss không giảm sau 15 epoch liên tiếp
BATCH_SIZE = 64 if GPU_VRAM_GB > 30 else 32   # giảm xuống 16 nếu báo CUDA out of memory
LEARNING_RATE = 1e-4
GRAD_CLIP = 5.0
SAMPLE_EVERY = 5
SAMPLE_SPEAKER = 0
SAMPLE_TEXT = 'Hôm nay trời trong xanh, gió nhẹ thổi qua những hàng cây ven đường.'


# ---------------------------- Dữ liệu (toàn bộ mel nằm sẵn trong RAM) ----------------------------
class VivosMelDataset(Dataset):
    def __init__(self, items):
        self.items = items
        self.lengths = [it['mel'].shape[-1] for it in items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        x = torch.tensor(text_to_ids(it['text'], CHAR2ID), dtype=torch.long)
        y = (it['mel'].float() - meta['mel_mean']) / meta['mel_std']
        return x, y, SPK2ID[it['spk']]


def collate(batch):
    x_lengths = torch.tensor([b[0].shape[0] for b in batch], dtype=torch.long)
    y_lengths = torch.tensor([b[1].shape[-1] for b in batch], dtype=torch.long)
    x = torch.zeros(len(batch), int(x_lengths.max()), dtype=torch.long)
    y = torch.zeros(len(batch), N_MELS, int(fix_len_compatibility(int(y_lengths.max()))))
    for i, (xi, yi, _) in enumerate(batch):
        x[i, :xi.shape[0]] = xi
        y[i, :, :yi.shape[-1]] = yi
    spks = torch.tensor([b[2] for b in batch], dtype=torch.long)
    return x, x_lengths, y, y_lengths, spks


class BucketBatchSampler:
    """Gom các câu có độ dài gần nhau vào cùng batch để giảm padding (nhanh hơn, ít tốn VRAM)."""

    def __init__(self, lengths, batch_size, bucket_batches=50):
        self.lengths, self.batch_size = lengths, batch_size
        self.chunk = batch_size * bucket_batches

    def __iter__(self):
        idx = list(range(len(self.lengths)))
        random.shuffle(idx)
        batches = []
        for s in range(0, len(idx), self.chunk):
            group = sorted(idx[s:s + self.chunk], key=lambda i: self.lengths[i])
            batches += [group[j:j + self.batch_size] for j in range(0, len(group), self.batch_size)]
        random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        n = len(self.lengths)
        return sum(math.ceil(min(self.chunk, n - s) / self.batch_size) for s in range(0, n, self.chunk))


train_set, val_set = VivosMelDataset(data['train']), VivosMelDataset(data['val'])
train_loader = DataLoader(train_set, batch_sampler=BucketBatchSampler(train_set.lengths, BATCH_SIZE),
                          collate_fn=collate, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=collate, pin_memory=True)

# ---------------------------- Mô hình: resume hoặc khởi tạo từ LJSpeech ----------------------------
start_epoch, global_step, best_val, best_epoch = 1, 0, float('inf'), 0
if os.path.exists(RESUME_CKPT):
    ckpt = torch.load(RESUME_CKPT, map_location='cpu', weights_only=False)
    hparams = ckpt['hparams']
    model = build_model(hparams)
    model.load_state_dict(ckpt['state_dict'])
    start_epoch, global_step, best_val = ckpt['epoch'] + 1, ckpt['step'], ckpt['best_val']
    best_epoch = ckpt.get('best_epoch', ckpt['epoch'])
    print(f'[RESUME] Tiep tuc tu epoch {start_epoch} (step {global_step}, best val {best_val:.4f})')
else:
    ckpt = None
    # PyTorch >= 2.6 mặc định weights_only=True, không đọc được hyper_parameters (omegaconf) của checkpoint
    # Lightning -> nạp với weights_only=False (checkpoint chính thức từ repo Matcha-TTS)
    lj_ckpt = torch.load(download(MATCHA_LJSPEECH_URL, 'matcha_ljspeech.ckpt'), map_location='cpu', weights_only=False)
    hparams = {k: v for k, v in lj_ckpt['hyper_parameters'].items() if k not in ('optimizer', 'scheduler')}
    hparams = OmegaConf.to_container(OmegaConf.create(hparams), resolve=True)
    hparams.update(n_vocab=len(meta['vocab']), n_spks=len(meta['speakers']), out_size=None,
                   data_statistics={'mel_mean': meta['mel_mean'], 'mel_std': meta['mel_std']})
    model = build_model(hparams)
    # Vocab tiếng Việt và speaker embedding khác LJSpeech -> chỉ nạp các tensor cùng kích thước
    own_state = model.state_dict()
    reused = {k: v for k, v in lj_ckpt['state_dict'].items()
              if k in own_state and own_state[k].shape == v.shape and k not in ('mel_mean', 'mel_std')}
    model.load_state_dict(reused, strict=False)
    print(f'[INIT] Tai su dung {len(reused)}/{len(own_state)} tensor tu checkpoint LJSpeech')
    del lj_ckpt

model = model.to(DEVICE).train()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_DTYPE == torch.float16)
if ckpt is not None:
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
del ckpt

vocoder, denoiser = load_vocoder()
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham so | batch {BATCH_SIZE} | '
      f'{len(train_loader)} step/epoch | AMP {AMP_DTYPE}')
print(f'Ke hoach: {EPOCHS} epoch = {EPOCHS * len(train_loader)} step (dung som neu val loss khong giam sau {PATIENCE} epoch)')


def run_batch(batch):
    x, x_lengths, y, y_lengths, spks = (t.to(DEVICE, non_blocking=True) for t in batch)
    with torch.autocast('cuda', dtype=AMP_DTYPE):
        dur_loss, prior_loss, diff_loss, _ = model(x, x_lengths, y, y_lengths, spks=spks)
    return dur_loss.float(), prior_loss.float(), diff_loss.float()


@torch.no_grad()
def evaluate():
    model.eval()
    totals = np.zeros(3)
    for batch in val_loader:
        totals += [l.item() for l in run_batch(batch)]
    model.train()
    return totals / len(val_loader)


def save_atomic(obj, path):
    # Ghi ra file tạm rồi đổi tên: nếu Colab ngắt giữa chừng, checkpoint cũ vẫn còn nguyên
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


# ---------------------------- Vòng lặp huấn luyện ----------------------------
for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        dur_loss, prior_loss, diff_loss = run_batch(batch)
        loss = dur_loss + prior_loss + diff_loss
        optimizer.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            print(f'[WARN] Loss khong hop le o step {global_step}, bo qua batch')
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        global_step += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}', dur=f'{dur_loss.item():.3f}',
                         prior=f'{prior_loss.item():.3f}', diff=f'{diff_loss.item():.3f}')

    val_dur, val_prior, val_diff = evaluate()
    val_loss = val_dur + val_prior + val_diff
    print(f'==> Epoch {epoch} | step {global_step} | {(time.time() - t0) / 60:.1f} phut | val loss {val_loss:.4f} '
          f'(dur {val_dur:.3f}, prior {val_prior:.3f}, diff {val_diff:.3f})')

    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        save_atomic({'hparams': hparams, 'state_dict': model.state_dict(), 'epoch': epoch, 'step': global_step},
                    MODEL_CKPT)
        print(f'[SAVE] Checkpoint tot nhat -> {MODEL_CKPT}')
    save_atomic({'hparams': hparams, 'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict(),
                 'scaler': scaler.state_dict(), 'epoch': epoch, 'step': global_step, 'best_val': best_val,
                 'best_epoch': best_epoch},
                RESUME_CKPT)

    if epoch % SAMPLE_EVERY == 0:
        wav = synthesize_vi(model, SAMPLE_TEXT, SAMPLE_SPEAKER, CHAR2ID, vocoder, denoiser)
        sf.write(f'{SAMPLES_DIR}/epoch{epoch:03d}.wav', wav, SAMPLE_RATE)
        display(Audio(wav, rate=SAMPLE_RATE))

    if epoch - best_epoch >= PATIENCE:
        print(f'[EARLY STOP] Validation loss khong giam tu epoch {best_epoch}, dung huan luyen.')
        break

print('\n[THANH CONG] Hoan tat huan luyen!')

## 6. Nghe Thử Giọng Tiếng Việt

Nạp `matcha_vi.pt` từ Drive. Có thể chạy độc lập sau khi kết nối lại: chạy mục 1, 2, 3, 4 rồi chạy cell này (bỏ qua mục 5). Đổi `SPEAKER_ID` để chọn giọng khác.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

ckpt = torch.load(MODEL_CKPT, map_location='cpu', weights_only=False)
infer_model = build_model(ckpt['hparams'])
infer_model.load_state_dict(ckpt['state_dict'])
infer_model = infer_model.to(DEVICE).eval()
print(f"[OK] Da nap {MODEL_CKPT} (epoch {ckpt['epoch']}, step {ckpt['step']})")
del ckpt

if 'vocoder' not in globals():
    vocoder, denoiser = load_vocoder()

# Danh sách người nói kèm giới tính (m = nam, f = nữ)
print('Nguoi noi:', ', '.join(f"{i}:{s}({meta['genders'].get(s, '?')})" for i, s in enumerate(meta['speakers'])))

SPEAKER_ID = 0
test_sentences = [
    'Hôm nay trời trong xanh, gió nhẹ thổi qua những hàng cây ven đường.',
    'Cảm ơn bạn đã lắng nghe, chúc bạn một ngày thật vui vẻ.',
]
for i, sentence in enumerate(test_sentences):
    print(f'Van ban: {sentence}')
    wav = synthesize_vi(infer_model, sentence, SPEAKER_ID, CHAR2ID, vocoder, denoiser,
                        n_timesteps=10, temperature=0.667, length_scale=1.0)
    out_path = f'{SAMPLES_DIR}/test_spk{SPEAKER_ID}_{i}.wav'
    sf.write(out_path, wav, SAMPLE_RATE)
    display(Audio(wav, rate=SAMPLE_RATE))